# Data Preparation for GWM-RNN Relation Prediction - FB15k-237

This notebook prepares the FB15k-237 dataset for training GWM-RNN on the Knowledge Graph Completion task.

## Dataset Overview
- **FB15k-237**: Subset of Freebase knowledge graph
- **Entities**: ~14,500 entities (movies, actors, locations, etc.)
- **Relations**: 237 relation types
- **Triples**: ~272k training, ~17k validation, ~20k test
- **Task**: Given `(head, relation, ?)`, predict the tail entity

## Processing Steps
1. Load raw triples from text files
2. Create entity and relation vocabularies
3. Generate inverse relations (doubles the data)
4. Encode entity/relation descriptions using Sentence-BERT
5. Save processed tensors for training

## 1. Setup and Configuration

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict
import json

# Paths
RAW_DATA_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\data\fb15k-237\raw")
OUTPUT_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\data\fb15k-237\processed\relation-prediction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # 384-dim embeddings
BATCH_SIZE = 256
DEVICE = 'cuda'

# Inverse relation settings
CREATE_INVERSE_RELATIONS = True  # Double the data by adding inverse triples

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Create inverse relations: {CREATE_INVERSE_RELATIONS}")

## 2. Load Raw Data

In [ ]:
def load_triples(file_path):
    """Load triples from tab-separated file."""
    triples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            h, r, t = line.strip().split('\t')
            triples.append((h, r, t))
    return triples

# Load all splits
train_triples = load_triples(RAW_DATA_DIR / 'train.txt')
valid_triples = load_triples(RAW_DATA_DIR / 'valid.txt')
test_triples = load_triples(RAW_DATA_DIR / 'test.txt')

print(f"Training triples: {len(train_triples):,}")
print(f"Validation triples: {len(valid_triples):,}")
print(f"Test triples: {len(test_triples):,}")
print(f"Total triples: {len(train_triples) + len(valid_triples) + len(test_triples):,}")

# Show examples
print("\nExample triples:")
for i, triple in enumerate(train_triples[:5], 1):
    print(f"{i}. {triple}")

## 3. Create Vocabularies and Inverse Relations

In [ ]:
def create_vocabularies(train_triples, valid_triples, test_triples, create_inverse=True):
    """
    Create entity and relation vocabularies.
    Optionally add inverse relations.
    """
    entities = set()
    relations = set()
    
    # Collect all entities and relations
    all_triples = train_triples + valid_triples + test_triples
    for h, r, t in all_triples:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    # Add inverse relations
    if create_inverse:
        original_relations = list(relations)
        for rel in original_relations:
            relations.add(rel + '_inv')
    
    # Create mappings
    entity2id = {ent: idx for idx, ent in enumerate(sorted(entities))}
    id2entity = {idx: ent for ent, idx in entity2id.items()}
    
    relation2id = {rel: idx for idx, rel in enumerate(sorted(relations))}
    id2relation = {idx: rel for rel, idx in relation2id.items()}
    
    return entity2id, id2entity, relation2id, id2relation

entity2id, id2entity, relation2id, id2relation = create_vocabularies(
    train_triples, valid_triples, test_triples, 
    create_inverse=CREATE_INVERSE_RELATIONS
)

print(f"Number of entities: {len(entity2id):,}")
print(f"Number of relations: {len(relation2id):,}")

if CREATE_INVERSE_RELATIONS:
    original_rels = [r for r in relation2id.keys() if not r.endswith('_inv')]
    inverse_rels = [r for r in relation2id.keys() if r.endswith('_inv')]
    print(f"  - Original relations: {len(original_rels)}")
    print(f"  - Inverse relations: {len(inverse_rels)}")

# Save vocabularies
with open(OUTPUT_DIR / 'entity2id.json', 'w') as f:
    json.dump(entity2id, f, indent=2)
with open(OUTPUT_DIR / 'relation2id.json', 'w') as f:
    json.dump(relation2id, f, indent=2)

print("\n✓ Vocabularies saved")

## 4. Load Entity and Relation Descriptions

In [ ]:
def load_entity_descriptions(file_path):
    """Load entity descriptions from mid2name or mid2description file."""
    descriptions = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                entity_id = parts[0]
                description = parts[1]
                if description.endswith("@en"):
                    description = description.removesuffix("@en")

                descriptions[entity_id] = description.strip("\"")
    return descriptions

# Try to load descriptions, fall back to names
entity_descriptions = {}
if (RAW_DATA_DIR / 'FB15k_mid2description.txt').exists():
    entity_descriptions = load_entity_descriptions(RAW_DATA_DIR / 'FB15k_mid2description.txt')
    print(f"Loaded {len(entity_descriptions):,} entity descriptions")
elif (RAW_DATA_DIR / 'FB15k_mid2name.txt').exists():
    entity_descriptions = load_entity_descriptions(RAW_DATA_DIR / 'FB15k_mid2name.txt')
    print(f"Loaded {len(entity_descriptions):,} entity names (using as descriptions)")
else:
    print("Warning: No entity descriptions found, will use entity IDs as text")

# Create entity texts (with fallback to entity ID)
entity_texts = {}
for entity in entity2id.keys():
    if entity in entity_descriptions:
        entity_texts[entity] = entity_descriptions[entity].strip('\"')
    else:
        # Clean up entity ID for readability
        clean_id = entity.replace('_', ' ').replace('/', ' ').strip()
        entity_texts[entity] = clean_id

print(f"\nEntity texts created: {len(entity_texts):,}")
print("\nExample entity texts:")
for entity in list(entity2id.keys())[:5]:
    print(f"  {entity} -> {entity_texts[entity]}")

In [ ]:
def create_relation_descriptions(relation2id):
    """
    Create human-readable descriptions for relations.
    Convert Freebase paths to natural language.
    """
    relation_texts = {}
    
    for relation in relation2id.keys():
        if relation.endswith('_inv'):
            # Inverse relation
            original = relation[:-4]
            # Clean the path
            parts = original.split('/')
            clean_parts = [p.replace('_', ' ') for p in parts if p]
            text = 'inverse of ' + ' '.join(clean_parts)
        else:
            # Original relation
            parts = relation.split('/')
            clean_parts = [p.replace('_', ' ') for p in parts if p]
            text = ' '.join(clean_parts)
        
        relation_texts[relation] = text
    
    return relation_texts

relation_texts = create_relation_descriptions(relation2id)

print(f"Relation texts created: {len(relation_texts):,}")
print("\nExample relation texts:")
for relation in list(relation2id.keys())[:10]:
    print(f"  {relation}")
    print(f"    -> {relation_texts[relation]}")

## 5. Generate Text Embeddings

Use Sentence-BERT to encode all entities and relations into dense vectors.

In [ ]:
# Install sentence-transformers if needed
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

import torch

# Load model
print(f"Loading embedding model: {EMBEDDING_MODEL}")
encoder = SentenceTransformer(EMBEDDING_MODEL)
encoder = encoder.to(DEVICE)

print(f"Model loaded on {DEVICE}")
print(f"Embedding dimension: {encoder.get_sentence_embedding_dimension()}")

In [ ]:
# Encode entities
print("Encoding entities...")
entity_list = sorted(entity2id.keys(), key=lambda x: entity2id[x])
entity_text_list = [entity_texts[e] for e in entity_list]

entity_embeddings = encoder.encode(
    entity_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"Entity embeddings shape: {entity_embeddings.shape}")
print(f"Memory size: {entity_embeddings.nbytes / (1024**2):.2f} MB")

In [ ]:
# Encode relations
print("Encoding relations...")
relation_list = sorted(relation2id.keys(), key=lambda x: relation2id[x])
relation_text_list = [relation_texts[r] for r in relation_list]

relation_embeddings = encoder.encode(
    relation_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"Relation embeddings shape: {relation_embeddings.shape}")
print(f"Memory size: {relation_embeddings.nbytes / (1024**2):.2f} MB")

## 6. Convert Triples to Tensor Format

In [ ]:
def convert_triples_to_ids(triples, entity2id, relation2id, add_inverse=True):
    """
    Convert triples from strings to IDs.
    Optionally add inverse triples.
    """
    id_triples = []
    
    for h, r, t in triples:
        h_id = entity2id[h]
        r_id = relation2id[r]
        t_id = entity2id[t]
        id_triples.append((h_id, r_id, t_id))
        
        # Add inverse triple
        if add_inverse:
            r_inv_id = relation2id[r + '_inv']
            id_triples.append((t_id, r_inv_id, h_id))
    
    return np.array(id_triples, dtype=np.int32)

# Convert all splits (only add inverse to training!)
train_ids = convert_triples_to_ids(train_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)
valid_ids = convert_triples_to_ids(valid_triples, entity2id, relation2id, add_inverse=False)
test_ids = convert_triples_to_ids(test_triples, entity2id, relation2id, add_inverse=False)

print(f"Training triples (with inverses): {len(train_ids):,}")
print(f"Validation triples: {len(valid_ids):,}")
print(f"Test triples: {len(test_ids):,}")
print(f"\nTotal processed triples: {len(train_ids) + len(valid_ids) + len(test_ids):,}")

if CREATE_INVERSE_RELATIONS:
    original_total = len(train_triples) + len(valid_triples) + len(test_triples)
    processed_total = len(train_ids) + len(valid_ids) + len(test_ids)
    print(f"Data augmentation: {original_total:,} -> {processed_total:,} ({processed_total/original_total:.1f}x)")

## 7. Create Ground Truth Dictionary

For filtered evaluation: Store all valid (h, r, t) triples to avoid penalizing correct predictions.

In [ ]:
def create_ground_truth_dict(train_ids, valid_ids, test_ids):
    """
    Create a dictionary mapping (h, r) -> set of valid tails.
    Used for filtered evaluation.
    """
    ground_truth = defaultdict(set)
    
    all_triples = np.concatenate([train_ids, valid_ids, test_ids], axis=0)
    
    for h, r, t in all_triples:
        ground_truth[(h, r)].add(t)
    
    # Convert to regular dict with lists
    ground_truth = {k: list(v) for k, v in ground_truth.items()}
    
    return ground_truth

ground_truth = create_ground_truth_dict(train_ids, valid_ids, test_ids)

print(f"Ground truth entries: {len(ground_truth):,}")
print(f"Average tails per (h, r): {np.mean([len(v) for v in ground_truth.values()]):.2f}")

# Save ground truth
# Convert keys from tuples to strings for JSON
# Convert numpy int32 to Python int for JSON serialization
ground_truth_json = {f"{h},{r}": [int(t) for t in tails] for (h, r), tails in ground_truth.items()}
with open(OUTPUT_DIR / 'ground_truth.json', 'w') as f:
    json.dump(ground_truth_json, f)

print("✓ Ground truth saved")

## 8. Save All Processed Data

In [ ]:
import torch

# Save embeddings
torch.save(torch.from_numpy(entity_embeddings), OUTPUT_DIR / 'entity_embeddings.pt')
torch.save(torch.from_numpy(relation_embeddings), OUTPUT_DIR / 'relation_embeddings.pt')

# Save triples
torch.save(torch.from_numpy(train_ids), OUTPUT_DIR / 'train_triples.pt')
torch.save(torch.from_numpy(valid_ids), OUTPUT_DIR / 'valid_triples.pt')
torch.save(torch.from_numpy(test_ids), OUTPUT_DIR / 'test_triples.pt')

# Save metadata
metadata = {
    'num_entities': len(entity2id),
    'num_relations': len(relation2id),
    'num_original_relations': len([r for r in relation2id if not r.endswith('_inv')]),
    'embedding_dim': entity_embeddings.shape[1],
    'embedding_model': EMBEDDING_MODEL,
    'has_inverse_relations': CREATE_INVERSE_RELATIONS,
    'train_size': len(train_ids),
    'valid_size': len(valid_ids),
    'test_size': len(test_ids),
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("="*70)
print("DATA PREPARATION COMPLETE")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles saved:")
print(f"  • entity_embeddings.pt ({entity_embeddings.shape})")
print(f"  • relation_embeddings.pt ({relation_embeddings.shape})")
print(f"  • train_triples.pt ({len(train_ids):,} triples)")
print(f"  • valid_triples.pt ({len(valid_ids):,} triples)")
print(f"  • test_triples.pt ({len(test_ids):,} triples)")
print(f"  • entity2id.json, relation2id.json")
print(f"  • ground_truth.json")
print(f"  • metadata.json")
print(f"\nDataset statistics:")
for key, value in metadata.items():
    print(f"  • {key}: {value}")
print("\n✓ Ready for training!")

## 9. Create context embeddings

In [ ]:
required_files = ['generate_context_embeddings.py']

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "context-aware-gwm-rnn"

!git clone {GITHUB_REPO} /kaggle/working/gwm
%cd /kaggle/working/gwm
!git checkout {BRANCH}
!git pull
%cd ../

# Copy files from repo to working directory
repo_path = "/kaggle/working/gwm/gwm-rnn/relation-prediction"

print(f"\nCopying files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /kaggle/working/
    print(f"✓ Copied {file}")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

In [ ]:
!python generate_context_embeddings.py \
    --data_dir /kaggle/working/dataset/ \
    --aggregation mean \
    --top_k 15

## 10. Data Statistics and Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Relation frequency analysis
relation_counts = defaultdict(int)
for h, r, t in train_ids:
    relation_counts[r] += 1

# Plot relation distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Top 20 relations
top_relations = sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:20]
rel_names = [id2relation[r] for r, _ in top_relations]
rel_counts = [c for _, c in top_relations]

axes[0].barh(range(len(rel_names)), rel_counts)
axes[0].set_yticks(range(len(rel_names)))
axes[0].set_yticklabels([r[:40] + '...' if len(r) > 40 else r for r in rel_names], fontsize=8)
axes[0].set_xlabel('Number of Triples')
axes[0].set_title('Top 20 Most Frequent Relations')
axes[0].invert_yaxis()

# 2. Distribution of relation frequencies
counts = list(relation_counts.values())
axes[1].hist(counts, bins=50, edgecolor='black')
axes[1].set_xlabel('Number of Triples')
axes[1].set_ylabel('Number of Relations')
axes[1].set_title('Distribution of Relation Frequencies')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'relation_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Most frequent relation: {id2relation[top_relations[0][0]]} ({top_relations[0][1]} triples)")
print(f"Least frequent relation: {id2relation[top_relations[-1][0]]} ({top_relations[-1][1]} triples)")
print(f"Average triples per relation: {np.mean(counts):.1f}")